[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_4_HigherOrder_Operators_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


<div dir="rtl" style="text-align:right">

# 📐 اللاب 4: المعادلات الخطية المتجانسة من الرتب العليا وأصل معادلة لابلاس
### *غرفة التحكم: لماذا تمنع معادلة لابلاس $\nabla^2\phi = 0$ وجود أي مصيدة محلية؟*
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)  
**المرجع في المذكرة:** الباب الأول، القسم 1.4 — **ترقيم المذكرة ص 34–57** *(صفحات ملف الـ PDF من 28 إلى 51؛ معادلة لابلاس في ص 39 بالترقيم المكتوب / PDF ص 33)*

---
### 🗺️ موقعك في سلسلة بناء محرك الملاحة (Rover Pipeline)
```
[اللاب 1–3: الميول، مسارات الانسياب، وهوامش الأمان] ──► 📍 [اللاب 4: معادلة لابلاس والجهد التوافقي] ──► [اللاب 5: المؤثرات القابلة للتحليل ومشكلة الاضمحلال] ──► [اللاب 6: الحل العددي SOR]
```

---
### 🧭 حجر رشيد (The Rosetta Stone): الربط بين السبورة ومحرك المركبة

| 1. رمز المذكرة (PDF pp. 28–51) | 2. المعنى الهندسي والفيزيائي | 3. الدالة الفعلية في المشروع (`rover_path_finder/`) |
| :--- | :--- | :--- |
| $D_x = \frac{\partial}{\partial x}, \quad D_y = \frac{\partial}{\partial y}$ | مؤثرات التفاضل الجزئي الجبرية | تحويل المعادلة التفاضلية إلى كثيرة حدود جبرية |
| $(D_x^2 + D_y^2)z = 0$ *(مثال 5 ص 39 / PDF ص 33)* | **معادلة لابلاس (Laplace's Equation):** انعدام التقوس الصافي؛ السطح مشدود كغشاء مطاطي | المعادلة الحاكمة لحقل الجهد الملاحي في [`solvers.py`](rover_path_finder/solvers.py) |
| $m^2 + 1 = 0 \implies m = \pm i$ | الجذور المميزة المركبة المترافقة لمعادلة لابلاس | تثبت أن الحل دالة توافقية ناعمة بلا قمم أو حفر داخلية |
| $\nabla^2 \phi \approx \frac{\phi_{E}+\phi_{W}+\phi_{S}+\phi_{N} - 4\phi_{C}}{h^2} = 0$ | قيمة أي نقطة تساوي تماماً متوسط جيرانها الأربعة | الدالة [`_avg(phi)`](rover_path_finder/solvers.py) في `rover_path_finder/solvers.py` |

---
## 1. الفكرة ببساطة: لماذا نحتاج إلى معادلة من الرتبة الثانية (معادلة لابلاس)؟

في اللاب 2 رأينا أن المركبة تتبع منحدر حقل الجهد $\phi(x, y)$ عبر معادلة لاجرانج (`descend`).
ولكن **كيف نصنع حقل الجهد $\phi(x,y)$ نفسه** بحيث يضمن للمركبة عدم الوقوع في أي "حفرة جهد وهمية" (Local Minimum) قبل وصولها للهدف؟

تخيل غشاءً مطاطياً مشدوداً على إطار:
* إذا ثبتنا ارتفاع الغشاء عند جدران الغرفة (أو العوائق) عند $\phi = 1$، وثبتنا نقطة الهدف فقط عند القاع $\phi = 0$:
* فإن الغشاء المطاطي في الفراغ بينهما يحقق معادلة التوازن التام:
  $$\frac{\partial^2 \phi}{\partial x^2} + \frac{\partial^2 \phi}{\partial y^2} = 0 \quad \Longleftrightarrow \quad (D_x^2 + D_y^2)\phi = 0$$
* هذه هي **معادلة لابلاس (Laplace's Equation)**!
* هندسياً، المشتقة الثانية $\frac{\partial^2 \phi}{\partial x^2}$ تقيس التقوس في اتجاه $x$، و $\frac{\partial^2 \phi}{\partial y^2}$ تقيس التقوس في اتجاه $y$.
* لكي يكون مجموعهما صفراً ($\nabla^2 \phi = 0$)، لا يمكن للسطح أن يتقوس للأعلى في الاتجاهين معاً في نفس النقطة! أي **يستحيل رياضياً وجود أي قاع محلي (Local Minimum)** داخل المجال المفتوح لأن القاع المحلي يتطلب $\phi_{xx} > 0$ و $\phi_{yy} > 0$ معاً (ومجموع رقمين موجبين لا يمكن أن يساوي صفراً!).

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab4

%matplotlib inline
print("تم تهيئة بيئة اللاب 4 بنجاح!")


<div dir="rtl" style="text-align:right">

---
## 2. التغطية الكاملة للقسم 1.4 في المذكرة: المعادلات المتجانسة وجميع حالات التكامل الخاص الـ 6 (PDF ص 28–51)

نسمي المعادلة التفاضلية الجزئية الخطية **متجانسة في الرتبة (Homogeneous of order $n$)** إذا كانت جميع المشتقات فيها من نفس الرتبة $n$:
$$(a_0 D_x^n + a_1 D_x^{n-1} D_y + \dots + a_n D_y^n) z = f(x, y) \quad \Longleftrightarrow \quad F(D_x, D_y) z = f(x, y)$$

يتكون الحل العام من جزأين: $z = z_1\text{ (الحل المكمل C.F.)} + z_2\text{ (التكامل الخاص P.I.)}$.

---
### أولاً: إيجاد الحل المكمل (Complementary Function $z_1$ — ص 35–39 / PDF ص 29–33)
نفرض حلاً من الشكل $z = \phi(y + mx)$، فتكون $D_x z = m\phi'$ و $D_y z = \phi'$. بالتعويض في $F(D_x, D_y)z = 0$ نحصل على **المعادلة المساعدة (Auxiliary Equation)** بوضع $D_x = m$ و $D_y = 1$:
$$F(m, 1) = a_0 m^n + a_1 m^{n-1} + \dots + a_n = 0$$

| حالة جذور المعادلة المساعدة $F(m, 1) = 0$ | شكل الحل المكمل $z_1$ في المذكرة |
| :--- | :--- |
| **الحالة 1: جذور حقيقية ومختلفة** $m_1 \neq m_2$ *(ص 36 / PDF ص 30)* | $z_1 = \phi_1(y + m_1 x) + \phi_2(y + m_2 x)$ |
| **الحالة 2: جذور مكررة ($k$ مرات)** $m_1 = \dots = m_k = m$ *(ص 37 / PDF ص 31)* | $z_1 = \phi_1(y + mx) + x\,\phi_2(y + mx) + \dots + x^{k-1}\phi_k(y + mx)$ |
| **الحالة 3: جذور مركبة** $m = \alpha \pm i\beta$ *(ص 39 / PDF ص 33)* | $z_1 = \phi_1(y + (\alpha + i\beta)x) + \phi_2(y + (\alpha - i\beta)x)$ |

---
### 📖 المثال المحوري في المذكرة: معادلة لابلاس (مثال 5 ص 39 / PDF ص 33)
**المسألة:** حل معادلة لابلاس ثنائية الأبعاد:
$$\frac{\partial^2 z}{\partial x^2} + \frac{\partial^2 z}{\partial y^2} = 0 \quad \Longleftrightarrow \quad (D_x^2 + D_y^2)z = 0$$
1. **المعادلة المساعدة:** نضع $D_x = m$ و $D_y = 1$:
   $$m^2 + 1 = 0 \implies m^2 = -1 \implies m = \pm i$$
2. **الحل العام التحليلي:**
   $$z = \phi_1(y + ix) + \phi_2(y - ix)$$
   حيث $\phi_1, \phi_2$ دالتان اختياريتان. (وإذا اخترنا مثلاً $\phi_1(w) = \frac{1}{2i}\sin(\pi i w)$ و $\phi_2(w) = -\frac{1}{2i}\sin(\pi i w)$، نحصل على الدالة التوافقية الحقيقية الشهيرة $\sin(\pi x)\sinh(\pi y)$!).

---
### ثانياً: إيجاد التكامل الخاص $z_2 = \frac{1}{F(D_x, D_y)} f(x, y)$ — الطريقة العامة والحالات الست المختصرة (PDF ص 34–51)

عندما يكون الطرف الأيمن $f(x, y) \neq 0$، تشرح المذكرة **طريقة عامة (General Method)** بالإضافة إلى **6 طرق مختصرة (Short Methods)** بحسب شكل الدالة $f(x, y)$:

#### 🧭 جدول القرار السريع لحالات التكامل الخاص الست (PDF ص 34–51)

| الحالة في المذكرة | شكل الطرف الأيمن $f(x, y)$ | القاعدة السريعة لحساب $z_2 = \frac{1}{F(D_x, D_y)}f(x,y)$ | ماذا لو صفر المقام ($F = 0$)؟ |
| :--- | :--- | :--- | :--- |
| **الطريقة العامة** *(ص 40 / PDF ص 34)* | أي دالة عامة (مثل $\sec^2(x+y)$) | حلل $F = \prod(D_x - m_i D_y)$ وحل كل عامل بمعادلة لاجرانج $\frac{dx}{1} = \frac{dy}{-m} = \frac{du}{f}$ | تعمل دائماً بدون استثناء! |
| **الحالة 1** *(ص 44 / PDF ص 38)* | $e^{ax + by}$ | استبدل $D_x \to a$ و $D_y \to b$: $z_2 = \frac{1}{F(a,b)}e^{ax+by}$ | إذا كان $F = (D_x - \frac{a}{b}D_y)^r G$: $z_2 = \frac{1}{G(a,b)}\frac{x^r}{r!}e^{ax+by}$ |
| **الحالة 2** *(ص 47 / PDF ص 41)* | $\sin(ax+by)$ أو $\cos(ax+by)$ | استبدل $D_x^2 \to -a^2,\; D_x D_y \to -ab,\; D_y^2 \to -b^2$ | حوّل لأسية بصيغة أويلر $\frac{e^{i\theta} \pm e^{-i\theta}}{2}$ وطبق رنين الحالة 1 |
| **الحالة 3** *(ص 51 / PDF ص 45)* | كثيرة حدود $x^a y^b$ | اسحب أصغر قوة لـ $D_x$ (أو $D_y$) عامل مشترك وفك $\frac{1}{1-\theta} = 1 + \theta + \theta^2 + \dots$ | التكامل $\frac{1}{D_x}$ يعني التكامل بالنسبة لـ $x$ |
| **الحالة 4** *(ص 53 / PDF ص 47)* | $e^{ax+by} V(x,y)$ | **الإزاحة الأسية:** أخرج $e^{ax+by}$ خارج المؤثر واستبدل $D_x \to D_x+a,\; D_y \to D_y+b$ | طبّق الحالة 1 أو 2 أو 3 على $V(x,y)$ بعد الإزاحة |
| **الحالة 5 و 6** *(ص 56 / PDF ص 50)* | أي دالة $g(ax+by)$ | إذا $F(a,b)\neq 0$: كامل $g(u)$ عدد $n$ مرات: $\frac{1}{F(a,b)}\int\dots\int g(u)du^n$ | **الحالة 6:** لـ $(bD_x - aD_y)^n z = g(ax+by)$: $z_2 = \frac{x^n}{n!\,b^n}g(ax+by)$ |

---
#### 🔹 شرح الطريقة العامة (General Method via Lagrange — ص 40–44 / PDF ص 34–38)
* **متى نستخدمها؟** عندما لا تكون $f(x, y)$ من الدوال القياسية البسيطة (أو عندما نريد الاشتقاق من المبادئ الأولى).
* **الفكرة:** نحلل المؤثر إلى عوامل خطية $F(D_x, D_y) = (D_x - m_1 D_y)(D_x - m_2 D_y)$، ونفرض $u_1 = (D_x - m_1 D_y)z$، فنحصل على معادلة من الرتبة الأولى:
  $$(D_x - m_2 D_y)u_1 = f(x, y) \iff \frac{\partial u_1}{\partial x} - m_2 \frac{\partial u_1}{\partial y} = f(x, y)$$
  نحلها بمعادلات لاجرانج المساعدة من **اللاب 2**: $\frac{dx}{1} = \frac{dy}{-m_2} = \frac{du_1}{f(x,y)}$، ثم نكرر نفس الخطوة لإيجاد $z$!
* **📖 مثال 1 في المذكرة (ص 41–43 / PDF ص 35–37):** حل $(D_x^2 - D_y^2)z = \sec^2(x + y)$.
  * نحلل $(D_x + D_y)(D_x - D_y)z = \sec^2(x + y)$. وبحل العاملين بالتتابع عبر لاجرانج نحصل على التكامل الخاص:
    $$\boxed{z_2 = \frac{x}{2}\tan(x + y)}$$

---
#### 🔹 الحالة 1: الدالة الأسية $f(x, y) = e^{ax + by}$ وحالة الرنين $F(a, b) = 0$ (ص 44–47 / PDF ص 38–41)
* **أ) إذا كان $F(a, b) \neq 0$:** نعوض مباشرة $D_x \to a$ و $D_y \to b$:
  $$\boxed{z_2 = \frac{1}{F(a, b)}e^{ax + by}}$$
  * **مثال 2 (ص 45 / PDF ص 39):** في $(D_x^2 - D_x D_y - 6D_y^2)z = e^{2x - 3y}$، لدينا $a = 2, b = -3$، فيكون $F(2, -3) = 4 - (2)(-3) - 6(-3)^2 = 4 + 6 - 54 = -44 \neq 0 \implies \boxed{z_2 = -\frac{1}{44}e^{2x - 3y}}$.
* **ب) ماذا لو كان المقام يساوي صفراً $F(a, b) = 0$ (حالة الرنين / الفشل)؟ (ص 45–47 / PDF ص 39–41)**
  إذا كان $F(a, b) = 0$، فهذا يعني أن $\left(D_x - \frac{a}{b}D_y\right)$ عامل من عوامل $F(D_x, D_y)$ مكرر $r$ من المرات:
  $$F(D_x, D_y) = \left(D_x - \frac{a}{b}D_y\right)^r G(D_x, D_y), \qquad G(a, b) \neq 0$$
  نعوض بـ $(a, b)$ في الجزء غير الصفري $G(a, b)$ فقط، أما العامل الصفري المكرر $r$ مرات فيتحول إلى **ضرب في $\frac{x^r}{r!}$**:
  $$\boxed{z_2 = \frac{1}{G(a, b)} \frac{x^r}{r!} e^{ax + by}}$$
  * **📖 مثال 3 في المذكرة (ص 46 / PDF ص 40):** حل $(D_x^2 - D_x D_y - 6D_y^2)z = e^{3x + y}$.
    1. هنا $a = 3, b = 1$. لو عوضنا مباشرة: $F(3, 1) = 9 - 3 - 6 = 0$! (حالة رنين لأن $e^{3x+y} = e^{3(x + y/3)}$ هي أصلًا ضمن الحل المكمل!).
    2. نحلل المؤثر: $D_x^2 - D_x D_y - 6D_y^2 = (D_x - 3D_y)^1 (D_x + 2D_y)$.
    3. إذن العامل الصفري هو $(D_x - 3D_y)$ بتكرار $r = 1$، والجزء غير الصفري هو $G(D_x, D_y) = D_x + 2D_y$ حيث $G(3, 1) = 3 + 2(1) = 5$.
    4. نطبق القانون مباشرة: $\boxed{z_2 = \frac{1}{5}\frac{x^1}{1!}e^{3x + y} = \frac{x}{5}e^{3x + y}}$.

> 📌 **تنبيه خطأ مطبعي في المذكرة (ص 49، مثال 5 / PDF ص 43):**
> في السطر الأخير من صفحة 49 بالترقيم المكتوب (PDF ص 43) عند حساب التكامل الخاص للدالة $e^{2x+3y}$ ($a=2, b=3$) للمؤثر $F = D_x^2 - 3D_x D_y + 2D_y^2$، كتبت المذكرة بالخطأ $F(1,1) = 4 - 18 + 18 = 4$. الأرقام المحسوبة ($2^2 - 3(2)(3) + 2(3)^2 = 4 - 18 + 18 = 4$) صحيحة تماماً لـ $(a,b)=(2,3)$، لكن الرمز المكتوب $F(1,1)$ هو خطأ مطبعي والصواب $\boxed{F(2, 3) = 4}$!

---
#### 🔹 الحالة 2: دوال الجيب وجيب التمام $\sin(ax+by)$ أو $\cos(ax+by)$ (ص 47–51 / PDF ص 41–45)
* **أ) في المؤثرات من الرتبة الثانية (بشرط عدم انعدام المقام):** نستبدل كل:
  $$\boxed{D_x^2 \to -a^2, \qquad D_x D_y \to -ab, \qquad D_y^2 \to -b^2}$$
  *(تنبيه الإشارات: السالب خارج التربيع دائماً! فإذا كان $a=2, b=-3$ فإن $D_x^2 \to -4$ و $D_y^2 \to -9$ بينما $D_x D_y \to -(2)(-3) = +6$).*
* **ب) ماذا لو صفر المقام في $\sin$ أو $\cos$؟ (مثال 6 ص 50–51 / PDF ص 44–45):**
  في $(D_x^2 + D_y^2)z = \cos(mx + my)$، لدينا $a = m, b = m$، فالتعويض يعطي $-m^2 + (-m^2)$ أو في حالة $(D_x^2 - D_y^2)z = \cos(mx+my)$ يعطي $-m^2 - (-m^2) = 0$!
  عندما ينعدم المقام (أو في المؤثرات من رتب فردية)، تضع المذكرة $\cos\theta = \frac{e^{i\theta} + e^{-i\theta}}{2}$ (أو $\sin\theta = \frac{e^{i\theta} - e^{-i\theta}}{2i}$) وتطبق قاعدة الرنين من **الحالة 1**!

---
#### 🔹 الحالة 3: كثيرات الحدود $f(x, y) = x^a y^b$ بطريقة المفكوك (ص 51–53 / PDF ص 45–47)
* **الفكرة:** نسحب العامل ذا أصغر أس لـ $D_x$ (أو $D_y$) من المقام ليصبح المقام على الصورة $D_x^k [1 - \theta(D_x, D_y)]$، ثم نرفعه للبسط بمتسلسلة ذات الحدين (المتسلسلة الهندسية):
  $$\boxed{\frac{1}{1 - \theta} = 1 + \theta + \theta^2 + \theta^3 + \dots, \qquad \frac{1}{1 + \theta} = 1 - \theta + \theta^2 - \theta^3 + \dots}$$
  لماذا تتوقف المتسلسلة ولا تستمر إلى ما لا نهاية؟ لأن $x^a y^b$ كثيرة حدود، فمشتقاتها العليا تصبح **صفراً** بعد عدد محدود من الخطوات! وتذكر أن $\frac{1}{D_x}$ معناه **التكامل بالنسبة لـ $x$** ($\int (\cdots)\,dx$).
* **📖 مثال 7 في المذكرة (ص 52 / PDF ص 46):** أوجد التكامل الخاص لـ $(D_x^2 - 2D_x D_y)z = x^3 y$.
  1. نسحب $D_x^2$ عاملاً مشتركاً من المقام:
     $$z_2 = \frac{1}{D_x^2\left(1 - \frac{2D_y}{D_x}\right)}(x^3 y) = \frac{1}{D_x^2}\left(1 - \frac{2D_y}{D_x}\right)^{-1}(x^3 y)$$
  2. نفك القوس بمتسلسلة هندسية (نحتاج فقط حتى المشتقة الأولى $D_y$ لأن $D_y^2(x^3 y) = 0$!):
     $$z_2 = \frac{1}{D_x^2}\left(1 + \frac{2D_y}{D_x} + \dots\right)(x^3 y) = \frac{1}{D_x^2}\left[x^3 y + \frac{2}{D_x}\underbrace{D_y(x^3 y)}_{x^3}\right]$$
  3. نكامل بالنسبة لـ $x$ ($\frac{1}{D_x}(x^3) = \frac{x^4}{4}$، ثم نكامل مرتين لوجود $\frac{1}{D_x^2}$ خارج القوس):
     $$z_2 = \frac{1}{D_x^2}\left(x^3 y + \frac{x^4}{2}\right) = \frac{1}{D_x}\left(\frac{x^4 y}{4} + \frac{x^5}{10}\right) = \boxed{\frac{x^5 y}{20} + \frac{x^6}{60}}$$

---
#### 🔹 الحالة 4: حاصل ضرب دالة أسية في أي دالة أخرى $f(x, y) = e^{ax + by} V(x, y)$ (ص 53–55 / PDF ص 47–50)
* **نظرية الإزاحة الأسية (Exponential Shift Rule):** يمكنك إخراج الدالة الأسية $e^{ax + by}$ إلى يسار المؤثر الكسري فوراً بشرط **إزاحة كل $D_x \to D_x + a$ وكل $D_y \to D_y + b$**:
  $$\boxed{\frac{1}{F(D_x, D_y)}\left[e^{ax + by} V(x, y)\right] = e^{ax + by} \frac{1}{F(D_x + a, D_y + b)} V(x, y)}$$
  وبعد إخراج $e^{ax+by}$، تصبح المسألة مجرد تأثير على $V(x,y)$ فنحلها بالحالة 1 أو 2 أو 3!
* **📖 مثال 10 في المذكرة (ص 54 / PDF ص 48):** حل $(D_x - D_y)^2 z = x e^{x + y}$.
  1. نخرج $e^{x + y}$ ($a = 1, b = 1$) ونزيح $D_x \to D_x + 1$ و $D_y \to D_y + 1$:
     $$(D_x + 1) - (D_y + 1) = D_x - D_y \implies z_2 = e^{x+y}\frac{1}{(D_x - D_y)^2}(x)$$
  2. الآن $V(x,y) = x^1 y^0$ لا يعتمد على $y$ إطلاقاً ($D_y(x) = 0$)، فيختزل المؤثر إلى $\frac{1}{D_x^2}(x)$ (تكامل $x$ مرتين بالنسبة لـ $x$):
     $$\boxed{z_2 = e^{x + y}\frac{1}{D_x^2}(x) = \frac{x^3}{6}e^{x + y}}$$
  *(وكذلك في **مثال 11 و 12 ص 54–55 / PDF ص 48–50** تطبق المذكرة الإزاحة الأسية مع $\sin(2x+3y)$ ومع $x^2 y$ بنفس الخطوات تماماً!).*

---
#### 🔹 الحالتان 5 و 6: عندما يكون الطرف الأيمن دالة عامة $f(x, y) = g(ax + by)$ (ص 56–57 / PDF ص 50–51)
* **الحالة 5 ($F(a, b) \neq 0$ — ص 56 / PDF ص 50):** نضع $u = ax + by$ ونكامل الدالة $g(u)$ عدد $n$ من المرات (حيث $n$ رتبة المؤثر):
  $$\boxed{z_2 = \frac{1}{F(a, b)} \underbrace{\int \dots \int}_{n\text{ مرات}} g(u)\,du^n}$$
* **الحالة 6 ($F(a, b) = 0$ حيث $F(D_x, D_y) = (b D_x - a D_y)^n$ — ص 56–57 / PDF ص 50–51):**
  إذا كان المؤثر من الصورة $(b D_x - a D_y)^n z = g(ax + by)$، فإن التعويض بـ $(a, b)$ يعطي $(ba - ab)^n = 0$! في هذه الحالة لا نكامل $g$ أصلاً، بل نضرب في $\frac{x^n}{n!\,b^n}$ مباشرة:
  $$\boxed{z_2 = \frac{x^n}{n!\,b^n}\,g(ax + by)}$$
  * **📖 مثال 14 و 15 في المذكرة (ص 57 / PDF ص 51):**
    * في **مثال 14:** $(2D_x - 3D_y)^2 z = \tan(3x + 2y)$، هنا $a = 3, b = 2, n = 2$ و $b D_x - a D_y = 2D_x - 3D_y \implies F(3, 2) = 0$. بتطبيق قانون الحالة 6 فوراً:
      $$\boxed{z_2 = \frac{x^2}{2!\cdot 2^2}\tan(3x + 2y) = \frac{x^2}{8}\tan(3x + 2y)}$$
    * في **مثال 15:** $(D_x - 2D_y)^3 z = (2x + y)^2$، هنا $a = 2, b = 1, n = 3 \implies \boxed{z_2 = \frac{x^3}{3!\cdot 1^3}(2x + y)^2 = \frac{x^3}{6}(2x + y)^2}$.

> 📌 **تنبيه خطأ مطبعي في المذكرة (ص 57، مثال 14 / PDF ص 51):**
> في حل المثال 14 في المذكرة، كُتب رمز المؤثر $F(D_x, D_x) = (2D_x - 3D_y)^2$ بحرف $D_x$ مرتين بين القوسين، وهو خطأ مطبعي بسيط والصواب $\boxed{F(D_x, D_y)}$.

</div>


<div dir="rtl" style="text-align:right">

---
## 3. 📝 تحدي الورقة والقلم 1 (جذور معادلة لابلاس — مثال 5 ص 39 / PDF ص 33)

**المسألة:** أوجد الجذور المميزة $m_1, m_2$ للمعادلة المساعدة لمعادلة لابلاس:
$$(D_x^2 + D_y^2)z = 0$$

في بايثون، تُكتب الوحدة التخيلية $i = \sqrt{-1}$ على الصورة `1j`، وتُكتب $-i$ على الصورة `-1j`.

<details>
<summary>💡 <b>تلميح (اضغط هنا إذا احتجت مساعدة)</b></summary>

ضع $D_x = m$ و $D_y = 1$ فتحصل على $m^2 + 1 = 0 \implies m = \pm i$. في بايثون اكتب `1j` و `-1j`.
</details>

</div>


In [ ]:
# خلية تقييم: جذور معادلة لابلاس
### START CODE HERE ### (≈ 2 lines of code)
# أدخل الجذور التخيلية لمعادلة لابلاس في بايثون:
m1 = None
m2 = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab4.check_paper_challenge_1(m1, m2)


<div dir="rtl" style="text-align:right">

---
## 4. 📝 تحدي الورقة والقلم 2 (التكامل الخاص P.I. — مثال 4 ص 48 / PDF ص 42)

**المسألة (من المذكرة ص 48 / PDF ص 42):** أوجد التكامل الخاص $z_2$ للمعادلة التفاضلية الجزئية غير المتجانسة في الطرف الأيمن:
$$(D_x^2 - D_x D_y - 6 D_y^2)z = \sin(2x - 3y)$$

**خطوات الحل على ورقتك:**
1. حدد معاملات الزاوية في $\sin(ax + by) = \sin(2x - 3y)$: لدينا $a = 2$ و $b = -3$.
2. عوض في المؤثر $\frac{1}{D_x^2 - D_x D_y - 6 D_y^2}$ باستخدام قاعدة الحالة الثانية:
   * $D_x^2 \to -a^2$
   * $D_x D_y \to -ab$
   * $D_y^2 \to -b^2$
3. احسب قيمة المقام العددي الناتج، ثم اكتب صيغة $z_2$.

<details>
<summary>💡 <b>تلميح 1 (حساب البدائل الثلاثة)</b></summary>

بما أن $a = 2$ و $b = -3$:
* $D_x^2 \to -(2)^2 = -4$
* $D_x D_y \to -(2)(-3) = +6$
* $D_y^2 \to -(-3)^2 = -9$
</details>

<details>
<summary>💡 <b>تلميح 2 (التعويض في المقام)</b></summary>

عوض بالأرقام في المقام $(D_x^2) - (D_x D_y) - 6(D_y^2)$:
$$(-4) - (6) - 6(-9) = -10 + 54 = 44$$
إذن التكامل الخاص هو مقلوب هذا العدد مضروباً في $\sin(2x - 3y)$.
</details>

</div>


In [ ]:
# خلية تقييم: التكامل الخاص P.I.
### START CODE HERE ### (≈ 1 line of code)
# اكتب صيغة التكامل الخاص بدلالة x و y:
student_pi = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab4.check_paper_challenge_2(student_pi)


<div dir="rtl" style="text-align:right">

---
## 5. 💻 التطبيق البرمجي ببايثون: حلال الجذور المميزة للمؤثرات التفاضلية

لحل أي معادلة متجانسة $(a_0 D_x^n + a_1 D_x^{n-1}D_y + \dots + a_n D_y^n)z = 0$ برمجياً، نمرر معاملات المعادلة المساعدة `[a_0, a_1, ..., a_n]` إلى دالة `np.roots(coeffs)` لإيجاد الجذور $m_1, \dots, m_n$ وتصنيف نوع المعادلة.

</div>


In [ ]:
# دالة تقييم برمجية: solve_operator_roots
def solve_operator_roots(coeffs: list[float]) -> list[complex]:
    # تحسب جذور المعادلة المساعدة F(m, 1) = 0 لقائمة المعاملات coeffs
    ### START CODE HERE ### (≈ 1 line of code)
    # استخدم np.roots(coeffs) لحساب الجذور:
    roots = None
    ### END CODE HERE ###
    return list(roots)

# تشغيل الاختبار الآلي:
test_lab4.test_characteristic_roots_solver(solve_operator_roots)


<div dir="rtl" style="text-align:right">

---
## 6. 🎨 الجسر إلى اللاب 6: كيف تتحول معادلة لابلاس التحليلية إلى دالة `_avg(phi)` في `solvers.py`؟

قد تتساءل: **إذا كان لدينا الحل التحليلي العام لمعادلة لابلاس $z = \phi_1(y + ix) + \phi_2(y - ix)$، فلماذا نحتاج إلى حلال عددي في اللاب 6؟**
* على مربع فارغ منتظم، يمكننا كتابة حل تحليلي مغلق مثل $\phi(x, y) = \frac{\sin(\pi x)\sinh(\pi y)}{\sinh(\pi)}$.
* لكن على خريطة المريخ، توجد **مئات الصخور والحفر بأشكال عشوائية غير منتظمة**! يستحيل إيجاد صيغة تحليلية واحدة تحقق شروط الحدود حول 10,000 عقدة صخرية عشوائية.
* الحل العبقري: عندما نقرب المشتقات الثانية على شبكة بكسل متساوية المسافات ($\Delta x = \Delta y = h$):
  $$\frac{\partial^2 \phi}{\partial x^2} + \frac{\partial^2 \phi}{\partial y^2} \approx \frac{\phi_{i,j+1} + \phi_{i,j-1} + \phi_{i+1,j} + \phi_{i-1,j} - 4\phi_{i,j}}{h^2} = 0$$
  فإن معادلة لابلاس $(D_x^2 + D_y^2)\phi = 0$ تعني ببساطة أن **قيمة كل نقطة داخلية تساوي متوسط جيرانها الأربعة (`_avg(phi)`)**:
  $$\phi_{i,j} = \frac{1}{4}\left(\phi_{\text{North}} + \phi_{\text{South}} + \phi_{\text{East}} + \phi_{\text{West}}\right)$$

شغّل الخلية التالية لترى السطح التحليلي التوافقي وتتحقق بنفسك أن دالة `_avg` الفعلية من ملف [`rover_path_finder/solvers.py`](rover_path_finder/solvers.py) تطابق السطح التحليلي!

</div>


In [ ]:
from rover_path_finder.solvers import _avg

x = np.linspace(0, 1, 50)
y = np.linspace(0, 1, 50)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.pi * X) * np.sinh(np.pi * Y) / np.sinh(np.pi)

# نختبر خاصية المتوسط الحسابي للجيران الأربعة على النقاط الداخلية
neighbor_diff = np.abs(_avg(Z)[1:-1, 1:-1] - Z[1:-1, 1:-1])

fig = plt.figure(figsize=(13, 5.5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.9)
ax1.set_title(r"1. Harmonic Surface $\nabla^2\phi = 0$ (Zero Local Minima)", fontsize=11)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel(r"$\phi(x, y)$")
plt.colorbar(surf, ax=ax1, shrink=0.55)

ax2 = fig.add_subplot(1, 2, 2)
im = ax2.imshow(neighbor_diff, origin='lower', extent=[x[1], x[-2], y[1], y[-2]], cmap='magma')
plt.colorbar(im, ax=ax2, label=r"$|\mathrm{\_avg}(\phi) - \phi|$")
ax2.set_title("2. Bridge to solvers.py: 4-Neighbor Average Residual", fontsize=11)
ax2.set_xlabel("x")
ax2.set_ylabel("y")

plt.tight_layout()
plt.show()

print(f"✅ أقصى فرق بين السطح التحليلي ومتوسط الجيران الأربعة _avg(Z): {neighbor_diff.max():.6f}")
print("📌 ملخص اللاب 4:")
print("  1) معادلة لابلاس (D_x^2 + D_y^2)z = 0 لها جذور مميزة مركبة m = ±i وحلولها دوال توافقية ملساء.")
print("  2) الدوال التوافقية تحقق خاصية المتوسط الحسابي، لذا يستحيل أن تحتوي على قاع محلي يحبس المركبة.")
